# **Configurations**

In [1]:
import sys

from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/College/FLEX-Med/flex-med

# Dependency Installation
!pip install fastapi uvicorn pyngrok nest-asyncio flwr["simulation"] -q

PROJECT_ROOT = "/content/drive/MyDrive/College/FLEX-Med/flex-med"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import flex_med
print("flex_med loaded from:", flex_med.__file__)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/College/FLEX-Med/flex-med
flex_med loaded from: /content/drive/MyDrive/College/FLEX-Med/flex-med/flex_med/__init__.py


# **API Definition**

In [ ]:
import os
import sys
import json
import logging
import subprocess
from typing import List, Tuple
import re
from datetime import datetime

from fastapi import FastAPI, BackgroundTasks, HTTPException
from pydantic import BaseModel
from starlette.middleware.cors import CORSMiddleware

# ==========================================
# 1. CONFIGURATION & PATHS
# ==========================================

# Import configuration from config.py
try:
    from flex_med.utils.config import (
        PROJECT_DIR,
        LOG_FILE_PATH,
        SIMULATION_LOG_PATH,
        DATA_JSON_PATH,
        PYPROJECT_PATH,
        RUN_SIMULATION_SHELL_FILE_PATH,
        SIMULATION_STATUS_JSON_FILE_PATH
    )
except ImportError:
    import sys
    sys.path.append(os.getcwd())
    from flex_med.utils.config import (
        PROJECT_DIR,
        LOG_FILE_PATH,
        SIMULATION_LOG_PATH,
        DATA_JSON_PATH,
        PYPROJECT_PATH,
        RUN_SIMULATION_SHELL_FILE_PATH,
        SIMULATION_STATUS_JSON_FILE_PATH
    )

os.makedirs(PROJECT_DIR, exist_ok=True)

# ==========================================
# 2. SERVER LOGGING SETUP
# ==========================================

logger = logging.getLogger("inference_logger")
logger.setLevel(logging.INFO)

if logger.hasHandlers():
    logger.handlers.clear()

file_handler = logging.FileHandler(LOG_FILE_PATH)
stream_handler = logging.StreamHandler(sys.stdout)

formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
file_handler.setFormatter(formatter)
stream_handler.setFormatter(formatter)

logger.addHandler(file_handler)
logger.addHandler(stream_handler)

logger.info("=== Flex-Med Orchestrator Started ===")

# ==========================================
# 3. CONFIGURATION VALIDATION
# ==========================================

def validate_client_config(client) -> Tuple[bool, str]:
    """
    Validate client configuration

    Returns:
        (is_valid, error_message)
    """
    valid_models = [
        'resnet18', 'resnet34', 'resnet50', 'resnet101', 'resnet152',
        'mobilenet_v2', 'mobilenet_v3_small', 'mobilenet_v3_large',
        'densenet121', 'densenet161', 'densenet169', 'densenet201',
        'efficientnet_b0', 'efficientnet_b1', 'efficientnet_b2', 'efficientnet_b3',
        'efficientnet_b4', 'efficientnet_b5', 'efficientnet_b6', 'efficientnet_b7',
        'vgg11', 'vgg13', 'vgg16', 'vgg19',
        'alexnet', 'squeezenet', 'inception_v3', 'googlenet'
    ]
    if client.model_type not in valid_models:
        return False, f"Invalid model type: {client.model_type}. Must be one of {valid_models}"

    if client.has_local_data:
        if not client.dataset_path:
            return False, f"Client {client.client_name} has_local_data=True but no dataset_path"
        if not os.path.exists(client.dataset_path):
            return False, f"Dataset path does not exist: {client.dataset_path}"

    if not client.model_path:
        return False, f"Client {client.client_name} has no model_path"

    model_dir = os.path.dirname(client.model_path)
    if not os.path.exists(model_dir):
        try:
            os.makedirs(model_dir, exist_ok=True)
            logger.info(f"Created model directory: {model_dir}")
        except Exception as e:
            return False, f"Cannot create model directory: {model_dir}, error: {e}"

    return True, ""

# ==========================================
# 4. CREATE RUN SCRIPT
# ==========================================

def create_run_script():
    """Creates the shell script that runs the Flower simulation."""
    script_content = f"""#!/bin/bash
cd "{PROJECT_DIR}"

echo "[Script] Starting Flex-Med Simulation at $(date)..."

# Only install the local package in editable mode (Fast & Lightweight)
pip install -e . -q

# Run Flower CLI directly
flwr run .
"""

    with open(RUN_SIMULATION_SHELL_FILE_PATH, "w") as f:
        f.write(script_content)

    os.chmod(RUN_SIMULATION_SHELL_FILE_PATH, 0o755)
    logger.info(f"Created run_simulation.sh at {RUN_SIMULATION_SHELL_FILE_PATH}")

    return RUN_SIMULATION_SHELL_FILE_PATH

SCRIPT_PATH = create_run_script()

# ==========================================
# 5. BACKGROUND TASK
# ==========================================

def run_simulation_task():
    """Runs the FL simulation in background."""
    logger.info("="*70)
    logger.info("Starting Flower Simulation in Background")
    logger.info("="*70)

    try:
        logger.info("\n[Background Task] Running Flower Simulation...")

        with open(SIMULATION_LOG_PATH, "w") as log_file:
            process = subprocess.run(
                ["bash", SCRIPT_PATH],
                stdout=log_file,
                stderr=log_file,
                text=True
            )

        return_code = process.returncode

        if return_code == 0:
            logger.info("\n" + "="*70)
            logger.info("✅ Federated Learning Simulation COMPLETED Successfully")
            logger.info("="*70)

            completion_status = {
                "status": "completed",
                "timestamp": datetime.now().isoformat(),
                "return_code": return_code,
                "log_path": SIMULATION_LOG_PATH
            }


            with open(SIMULATION_STATUS_JSON_FILE_PATH, 'w') as f:
                json.dump(completion_status, f, indent=2)

            logger.info(f"✅ Status saved to: {SIMULATION_STATUS_JSON_FILE_PATH}")

        else:
            logger.error("\n" + "="*70)
            logger.error(f"❌ Federated Learning Simulation FAILED")
            logger.error(f"   Return Code: {return_code}")
            logger.error("="*70)

            error_status = {
                "status": "failed",
                "timestamp": datetime.now().isoformat(),
                "return_code": return_code,
                "error": "Simulation failed",
                "log_path": SIMULATION_LOG_PATH
            }


            with open(SIMULATION_STATUS_JSON_FILE_PATH, 'w') as f:
                json.dump(error_status, f, indent=2)

    except Exception as e:
        logger.error(f"\n❌ Simulation error: {e}")
        import traceback
        logger.error(traceback.format_exc())

        try:
            error_status = {
                "status": "exception",
                "timestamp": datetime.now().isoformat(),
                "error": str(e),
                "traceback": traceback.format_exc(),
                "log_path": SIMULATION_LOG_PATH
            }

            with open(SIMULATION_STATUS_JSON_FILE_PATH, 'w') as f:
                json.dump(error_status, f, indent=2)
        except:
            pass

# ==========================================
# 6. PYDANTIC MODELS
# ==========================================

class ClientConfig(BaseModel):
    id: int
    created_at: str
    client_name: str
    model_type: str
    dataset_path: str | None
    status: str
    has_local_data: bool
    model_path: str
    metrics: str

# ==========================================
# 7. FASTAPI APP
# ==========================================

app = FastAPI(title="Flex-Med Orchestrator")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ==========================================
# 8. ENDPOINTS
# ==========================================

@app.get("/")
def read_root():
    return {"message": "Flex-Med Orchestrator is running"}

def update_data_json(clients: List[ClientConfig], path: str = DATA_JSON_PATH):
    """Save client configuration to data.json"""
    try:
        client_dicts = [c.dict() for c in clients]
        os.makedirs(os.path.dirname(path), exist_ok=True)

        with open(path, "w") as f:
            json.dump(client_dicts, f, indent=2)

        logger.info(f"✓ Updated data.json with {len(clients)} clients")
        return True
    except Exception as e:
        logger.error(f"Failed to update data.json: {e}")
        return False

def update_pyproject_toml(num_clients: int, path: str = PYPROJECT_PATH):
    """Update pyproject.toml with correct num-supernodes"""
    try:
        if not os.path.exists(path):
            logger.error(f"pyproject.toml not found at {path}")
            return False

        with open(path, 'r') as f:
            content = f.read()

        pattern = r'(options\.num-supernodes\s*=\s*)\d+'

        if not re.search(pattern, content):
            logger.warning("num-supernodes not found in pyproject.toml")
            return False

        old_match = re.search(r'options\.num-supernodes\s*=\s*(\d+)', content)
        old_value = old_match.group(1) if old_match else "unknown"

        new_content = re.sub(pattern, f'\\g<1>{num_clients}', content)

        with open(path, 'w') as f:
            f.write(new_content)

        logger.info(f"✓ Updated pyproject.toml: num-supernodes {old_value} → {num_clients}")
        return True

    except Exception as e:
        logger.error(f"Failed to update pyproject.toml: {e}")
        return False

@app.post("/start_fl")
async def start_fl(
    clients: List[ClientConfig],
    background_tasks: BackgroundTasks,
):
    """
    Start federated learning simulation with dynamic client configuration.

    This endpoint:
    1. Validates client configurations
    2. Saves client config to data.json
    3. Updates pyproject.toml with correct num-supernodes
    4. Triggers the FL simulation in background
    """

    num_clients = len(clients)
    logger.info(f"=" * 70)
    logger.info(f"Received FL start request for {num_clients} clients")
    logger.info(f"=" * 70)

    # Log client details
    for i, client in enumerate(clients):
        data_status = "Has Data" if client.has_local_data else "Free Rider"
        logger.info(f"  Client {i}: {client.client_name:20} | {client.model_type:15} | {data_status}")

    # ===== VALIDATION: CHECK CLIENT CONFIGS =====
    logger.info("\n[Endpoint] Validating client configurations...")

    validation_errors = []
    for i, client in enumerate(clients):
        is_valid, error_msg = validate_client_config(client)
        if not is_valid:
            validation_errors.append(f"Client {i}: {error_msg}")

    if validation_errors:
        error_detail = "\n".join(validation_errors)
        logger.error(f"Configuration validation failed:\n{error_detail}")
        raise HTTPException(
            status_code=400,
            detail=f"Invalid client configuration:\n{error_detail}"
        )

    logger.info("✓ All client configurations validated")

    # ===== STEP 1: UPDATE DATA.JSON =====
    logger.info("\n[Endpoint] Step 1: Updating data.json...")

    if not update_data_json(clients, DATA_JSON_PATH):
        raise HTTPException(
            status_code=500,
            detail="Failed to save client configuration to data.json"
        )

    # ===== STEP 2: UPDATE PYPROJECT.TOML =====
    logger.info("\n[Endpoint] Step 2: Updating pyproject.toml...")

    if not update_pyproject_toml(num_clients, PYPROJECT_PATH):
        logger.warning("Failed to update pyproject.toml, but continuing...")

    # ===== STEP 3: CREATE RUN SCRIPT =====
    logger.info("\n[Endpoint] Step 3: Creating run script...")
    global SCRIPT_PATH
    SCRIPT_PATH = create_run_script()

    # ===== STEP 4: TRIGGER SIMULATION =====
    logger.info("\n[Endpoint] Step 4: Starting FL simulation...")
    logger.info(f"  - Number of clients: {num_clients}")
    logger.info(f"  - Active participants: {sum(1 for c in clients if c.has_local_data)}")
    logger.info(f"  - Free riders: {sum(1 for c in clients if not c.has_local_data)}")
    logger.info(f"=" * 70 + "\n")

    # Trigger the background task
    background_tasks.add_task(run_simulation_task)

    return {
        "status": "accepted",
        "message": f"Federated simulation started with {num_clients} clients",
        "num_clients": num_clients,
        "active_participants": sum(1 for c in clients if c.has_local_data),
        "free_riders": sum(1 for c in clients if not c.has_local_data),
        "config_updated": {
            "data_json": DATA_JSON_PATH,
            "pyproject_toml": PYPROJECT_PATH
        },
        "log_path": SIMULATION_LOG_PATH
    }

print("✅ Flex-Med Orchestrator Ready")

2026-01-18 08:46:40,457 - INFO - === Flex-Med Orchestrator Started ===


INFO:inference_logger:=== Flex-Med Orchestrator Started ===


2026-01-18 08:46:40,529 - INFO - Created run_simulation.sh at /content/drive/MyDrive/College/FLEX-Med/flex-med/flex_med/utils/run_simulation.sh


INFO:inference_logger:Created run_simulation.sh at /content/drive/MyDrive/College/FLEX-Med/flex-med/flex_med/utils/run_simulation.sh


✅ Flex-Med Orchestrator Ready (with Supabase metrics support)


# **NGROK Setup**

In [ ]:
import nest_asyncio
import uvicorn
import subprocess
import threading
from pyngrok import ngrok
from pyngrok.process import kill_process
from fastapi import FastAPI, BackgroundTasks, HTTPException
from starlette.middleware.cors import CORSMiddleware

# 0. Cleanup
try:
    ngrok.disconnect()
    kill_process()
except:
    pass

# 1. Auth & Tunnel
# REPLACE WITH YOUR ACTUAL TOKEN
AUTH_TOKEN = "2nI3IUybQHGO30McAH7x4UJRiND_695D6z3DbKjWLrq7N1ATy"
ngrok.set_auth_token(AUTH_TOKEN)

http_tunnel = ngrok.connect(8000)
public_url = http_tunnel.public_url

print(f"🌐 API URL: {public_url}")
print(f"🚀 POST to start FL: {public_url}/start_fl")

# 2. Run Uvicorn
nest_asyncio.apply()

def run_uvicorn():
    uvicorn.run(app, port=8000, host="0.0.0.0")

thread = threading.Thread(target=run_uvicorn)
thread.start()

# **View Training Logs**

In [ ]:
!tail -f "/content/drive/MyDrive/College/FLEX-Med/flex-med/simulation_run.log"

# **Kill process**

In [ ]:
# Kill any process running on port 8000
!fuser -k 8000/tcp

print("Attempted to kill processes on port 8000. Please re-run the FastAPI server cell now.")

# **FL Simulation Evaluation**

In [ ]:
# ==========================================
# FL EVALUATION - GENERATE VISUALIZATIONS
# ==========================================
# This cell generates comprehensive graphical visualizations
# of the FL training metrics stored in cnmc_data.json

import sys
import os

# Ensure the project is in path
PROJECT_ROOT = "/content/drive/MyDrive/College/FLEX-Med/flex-med"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Import the visualization module
from flex_med.utils.fl_evaluation import (
    generate_all_visualizations,
    print_metrics_summary,
    DATA_JSON_PATH,
    GRAPHS_OUTPUT_DIR
)

# Print summary first
print_metrics_summary()

# Generate all visualizations
print("\n")
saved_files = generate_all_visualizations(
    data_path=DATA_JSON_PATH,
    output_dir=GRAPHS_OUTPUT_DIR,
    show_plots=True  # Set to False if you only want to save, not display
)

print(f"\n📊 All visualizations saved to: {GRAPHS_OUTPUT_DIR}")

# **View Individual Graphs (Optional)**
Run the cell below to view specific graphs individually.

In [ ]:
# ==========================================
# VIEW INDIVIDUAL GRAPHS
# ==========================================
# Uncomment the visualization you want to view

from flex_med.utils.fl_evaluation import (
    load_cnmc_data,
    plot_metric_across_rounds,
    plot_improvement_analysis,
    plot_improvement_over_rounds,
    plot_training_convergence,
    plot_confusion_matrix_comparison,
    plot_class_balance_analysis,
    plot_summary_dashboard,
    DATA_JSON_PATH,
    GRAPHS_OUTPUT_DIR
)

clients = load_cnmc_data(DATA_JSON_PATH)

# Uncomment the visualization you want to generate:

# Summary dashboard (recommended first view)
# plot_summary_dashboard(clients, GRAPHS_OUTPUT_DIR, show_plot=True)

# Individual metric progressions
# plot_metric_across_rounds(clients, 'accuracy', GRAPHS_OUTPUT_DIR, show_plot=True)
# plot_metric_across_rounds(clients, 'f1_score', GRAPHS_OUTPUT_DIR, show_plot=True)
# plot_metric_across_rounds(clients, 'precision', GRAPHS_OUTPUT_DIR, show_plot=True)
# plot_metric_across_rounds(clients, 'recall', GRAPHS_OUTPUT_DIR, show_plot=True)
# plot_metric_across_rounds(clients, 'loss', GRAPHS_OUTPUT_DIR, show_plot=True)
# plot_metric_across_rounds(clients, 'roc_auc', GRAPHS_OUTPUT_DIR, show_plot=True)

# Improvement analysis
# plot_improvement_analysis(clients, GRAPHS_OUTPUT_DIR, show_plot=True)
# plot_improvement_over_rounds(clients, GRAPHS_OUTPUT_DIR, show_plot=True)

# Training convergence (distillation & training loss)
# plot_training_convergence(clients, GRAPHS_OUTPUT_DIR, show_plot=True)

# Confusion matrix comparison (Pre-FL vs Post-FL)
# plot_confusion_matrix_comparison(clients, GRAPHS_OUTPUT_DIR, show_plot=True)

# Class balance analysis
# plot_class_balance_analysis(clients, GRAPHS_OUTPUT_DIR, show_plot=True)

print("Uncomment the visualization function you want to run above.")